# 1.6 Laplace And Time-Domain Modeling

Frequency-domain sweeps reconstruct time-domain traces over a finite period set by `df = 1 / T_max`. Energy that arrives after that period appears as periodic wrap-around in the reconstructed gather. Laplace-domain modeling adds exponential damping during the solve, which can reduce wrap-around before reconstruction.

This tutorial builds a shallow-water-like coupled model with very slow elastic sediments, then compares a standard time-domain sweep with Laplace-domain sweeps whose amplitudes decay by factors of 10 and 100 over one reconstruction period.

## What The Damping Controls

| Control | Meaning | Use |
| --- | --- | --- |
| `T_max` | Period of the reconstructed time signal. | Longer periods reduce wrap-around by increasing the modeled frequency count. |
| `damping_factor` | Amplitude decay over one period before compensation. | Convenient Laplace-domain control. `10` means the damped signal is one tenth as large at `T_max`. |
| `laplace` | Direct complex-frequency offset. | Advanced control equivalent to `-log(damping_factor) / (2*pi*T_max)`. |
| `traces.ld(...)` | Damped Laplace-domain reconstruction. | Diagnostic view with no amplitude compensation. |
| `traces.td(...)` | Physical time-domain reconstruction. | Applies amplitude compensation automatically when Laplace metadata are present. |

Damping is not a free improvement. Strong damping reduces periodic contamination, but the compensation step amplifies late-time amplitudes, interpolation error, and numerical noise. Use the smallest damping that fixes the specific wrap-around problem you can see.

## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import frequensolve as fs

u = fs.ureg

## Build A Slow-Sediment Model

The water layer is acoustic. The sediments are elastic and intentionally slow in shear velocity, so interface and converted waves can arrive later than the reconstruction period. This makes wrap-around easy to see without a large model.

In [ ]:
project = fs.Project(
    name="project",
    pretty_name="laplace_time_domain",
    path="./scratch/tutorials/laplace_time_domain",
    log_level="INFO",
    log_to_console=True,
)

sim = project.new_simulation(
    name="laplace_time_domain",
    physics="coupled_aep",
    dimension=2,
    units={"length": "km", "velocity": "km/s", "density": "g/cm^3"},
)

model = fs.LayeredModel(name="model", dimension=2, x_limits=[0.0, 1.2])
model.add_surface(name="top", depth=0.0 * u.km)
model.add_layer(
    name="water",
    physics="acoustic",
    properties={"Vp": 1.48, "Rho": 1.0, "Qp": 200.0},
)
model.add_surface(name="seafloor", depth=0.06 * u.km)
model.add_layer(
    name="soft_sediment",
    physics="elastic",
    properties={"Vp": 1.55, "Vs": 0.18, "Rho": 1.65, "Qp": 80.0, "Qs": 40.0},
)
model.add_surface(name="stiffer_sediment", depth=0.28 * u.km)
model.add_layer(
    name="halfspace",
    physics="elastic",
    properties={"Vp": 2.25, "Vs": 0.75, "Rho": 2.05, "Qp": 100.0, "Qs": 60.0},
)
model.add_surface(name="bottom", depth=0.55 * u.km)
sim += model

model.plot("vs", figsize=(8, 3), aspect="equal")

## Mesh, Boundaries, And Acquisition

The source is a vertical force just below the seafloor, and the receiver line measures vertical velocity along the shallow sediment. The water layer is still present in the model, so the run includes acoustic-elastic coupling at the seafloor.

In [ ]:
sim += model.hex_mesh_generator([12, 6])
sim.mesh.set_adapt(elems_per_wave=1.5, order=4, f_low=1.0, f_high=12.0)
sim.mesh.set_source_grading(d1=0.04, factor=2.0)

sim += fs.BoundaryCondition(conditions=["free"], boundaries=["z_min"])
sim += fs.BoundaryCondition(
    conditions=["pml"],
    boundaries=["x_min", "x_max", "z_max"],
    pml_wavelengths=0.75,
    pml_reflection=1.0e-3,
)

acq = fs.Acquisition()
acq.add_source_group(kind="vector", coords=[[0.18, 0.08]], direction=[0.0, 1.0])

geophone = fs.ReceiverNode(name="seafloor_geophone")
geophone.add_component(name="v_z", field="velocity", direction=[0.0, 1.0])

receiver_coords = [[x, 0.07] for x in np.linspace(0.05, 1.15, 181)]
acq.add_receiver_group(name="seafloor", device=geophone, coords=receiver_coords)
sim += acq

sim += fs.Discretization()
sim += fs.SolverConfig(tolerance=1.0e-4, grids=3)

project.save()

## Choose The Time Period And Damping

`T_max` sets the period of the inverse FFT reconstruction. The direct Laplace offset for a requested damping factor is shown here for reference; most users should use `damping_factor` on the job.

In [ ]:
T_MAX = 1.2
F_MAX = 12.0
WAVELET_F = 4.0

def laplace_for_factor(factor, period=T_MAX):
    return -np.log(float(factor)) / (2.0 * np.pi * period)

{
    "df_hz": 1.0 / T_MAX,
    "laplace_for_10": laplace_for_factor(10.0),
    "laplace_for_100": laplace_for_factor(100.0),
}

## Run Standard And Laplace-Domain Sweeps

The run cell is strict. If a local solver is not configured, or if the coupled solve fails, the notebook should stop here with logs and result directories available for inspection.

In [ ]:
cases = [
    ("standard", None, "Standard time domain"),
    ("damping_10", 10.0, "Damping factor 10"),
    ("damping_100", 100.0, "Damping factor 100"),
]

def run_case(name, damping_factor):
    kwargs = {}
    if damping_factor is not None:
        kwargs["damping_factor"] = damping_factor

    site = fs.LocalSite(shutdown_on_completion=True, verbose=True)
    job = fs.TimeDomainJob(
        name=name,
        simulation=sim,
        f_min=0.0,
        f_max=F_MAX,
        T_max=T_MAX,
        **kwargs,
    )
    result = site.submit(job).wait()
    return result.traces(upscale=4)

trace_sets = {
    name: run_case(name, damping_factor)
    for name, damping_factor, _title in cases
}

trace_sets["standard"].summary

## Reconstruct Compensated Time-Domain Gathers

For the Laplace-domain jobs, `traces.td(...)` applies the amplitude compensation automatically. The equivalent uncompensated diagnostic is `traces.ld(...)`, which is useful for confirming how much damping was applied before compensation.

In [ ]:
GROUP = "seafloor"
COMPONENT = "v_z"

def read_td(traces):
    source = traces.sources(GROUP)[0]
    return traces.td(
        GROUP,
        COMPONENT,
        source,
        fs.RickerWavelet(f=WAVELET_F),
        upscale=4,
        T_max=T_MAX,
    )

def read_ld(traces):
    source = traces.sources(GROUP)[0]
    return traces.ld(
        GROUP,
        COMPONENT,
        source,
        fs.RickerWavelet(f=WAVELET_F),
        upscale=4,
        T_max=T_MAX,
    )

gathers = {name: read_td(traces) for name, traces in trace_sets.items()}
laplace_gathers = {
    name: read_ld(trace_sets[name])
    for name in ("damping_10", "damping_100")
}

{name: gather.attrs for name, gather in gathers.items()}

## Compare Wrap-Around In The Physical Time Domain

All three panels use the same plotting scale. The damped runs should reduce periodic contamination from arrivals that would otherwise re-enter near the start of the time window.

In [ ]:
scale = max(
    np.nanpercentile(np.abs(np.real(gather.values)), 99.5)
    for gather in gathers.values()
)
scale = max(scale, 1.0e-12)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, _damping_factor, title) in zip(axes, cases):
    fs.plot_gather(
        gathers[name],
        ax=ax,
        A=scale,
        cmap="gray",
        title=title,
        T_max=T_MAX,
    )
fig.tight_layout()

## Inspect The Damped Diagnostic View

`ld` is not a physical-amplitude gather. It shows the damped signal before compensation, which is a good diagnostic when deciding whether a damping factor is stronger than necessary.

In [ ]:
ld_scale = max(
    np.nanpercentile(np.abs(np.real(gather.values)), 99.5)
    for gather in laplace_gathers.values()
)
ld_scale = max(ld_scale, 1.0e-12)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, name in zip(axes, ("damping_10", "damping_100")):
    fs.plot_gather(
        laplace_gathers[name],
        ax=ax,
        A=ld_scale,
        cmap="gray",
        title=f"{name}: uncompensated ld",
        T_max=T_MAX,
    )
fig.tight_layout()

## Compare One Receiver Trace

A single far-offset trace makes the compensation tradeoff easier to inspect. The strongest damping usually cleans up periodic leakage most aggressively, but late-time amplitudes are also the most amplified by compensation.

In [ ]:
receiver = gathers["standard"].coords["receiver"].values[-1]

fig, ax = plt.subplots(figsize=(9, 4))
for name, _damping_factor, title in cases:
    trace = gathers[name].sel(receiver=receiver)
    ax.plot(trace.coords["time"].values, np.real(trace.values), label=title)

ax.axhline(0.0, color="0.2", linewidth=0.8)
ax.set_xlim(0.0, T_MAX)
ax.set_xlabel("Time (s)")
ax.set_ylabel(COMPONENT)
ax.legend()
ax.set_title(f"Far-offset receiver {receiver}")
fig.tight_layout()

## Review Checklist

| Check | What to look for |
| --- | --- |
| Standard gather | Periodic energy near early time may indicate arrivals wrapping from the end of the reconstruction period. |
| Damping factor 10 | Often enough to reduce wrap-around while keeping compensation moderate. |
| Damping factor 100 | Cleaner periodic behavior can come with larger late-time amplification. Treat small late arrivals cautiously. |
| `ld` view | Confirms the raw damped result before amplitude compensation. |
| Production runs | Prefer increasing `T_max` when cost allows; use Laplace damping when the longer period is too expensive or when a targeted wrap-around problem remains. |